In [13]:
from datasets import load_dataset
from mteb.tasks.clustering.eng.ave_dataset_clustering import DATASET

train_dataset = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(50_000))

train_dataset = train_dataset.remove_columns("idx")

In [2]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

In [3]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import losses

embedding_model = SentenceTransformer("bert-base-uncased")

train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)

C:\Users\Миро\AppData\Local\Temp\ipykernel_13444\3854418656.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\Миро\AppData\Local\Temp\ipykernel_13444\3854418656.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
The

In [4]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

C:\Users\Миро\AppData\Local\Temp\ipykernel_13444\1746085209.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


In [5]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir="base_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)

C:\Users\Миро\AppData\Local\Temp\ipykernel_13444\2364888414.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


In [6]:
from sentence_transformers.trainer import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

trainer.train()

C:\Users\Миро\AppData\Local\Temp\ipykernel_13444\1213040465.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Column 'hypothesis' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.079172
200,0.952401
300,0.894772
400,0.855036
500,0.828428
600,0.835915
700,0.817749
800,0.797184
900,0.776992
1000,0.768131


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.8174072211931245, metrics={'train_runtime': 838.4125, 'train_samples_per_second': 59.637, 'train_steps_per_second': 1.864, 'total_flos': 0.0, 'train_loss': 0.8174072211931245, 'epoch': 1.0})

In [7]:
evaluator(embedding_model)

{'pearson_cosine': 0.5221325022623429, 'spearman_cosine': 0.5899377291863802}

In [10]:
import mteb

tasks = mteb.get_tasks(tasks=["Banking77Classification.v2"])
evaluation = mteb.MTEB(tasks=tasks)
results = evaluation.run(embedding_model)

C:\Users\Миро\AppData\Local\Temp\ipykernel_13444\515140376.py:4: DeprecationWarning: MTEB is deprecated and will be removed in future versions. Please use the `mteb.evaluate` function instead.
  evaluation = mteb.MTEB(tasks=tasks)
D:\python projects\hands_on_lmm\.venv\Lib\site-packages\mteb\models\model_meta.py:779: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
D:\python projects\hands_on_lmm\.venv\Lib\site-packages\mteb\models\model_meta.py:741: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification.v2, t2c

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/294k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/91.4k [00:00<?, ?B/s]

D:\python projects\hands_on_lmm\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
for result in results:
    print(result.task_name)
    print(result.scores)

Banking77Classification.v2
{'test': [{'scores_per_experiment': [{'accuracy': 0.5767230169050716, 'f1': 0.5763598646802979, 'f1_weighted': 0.57642325522278, 'precision': 0.5901267291623715, 'precision_weighted': 0.5901896122060677, 'recall': 0.5766483516483516, 'recall_weighted': 0.5767230169050716, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6180104031209362, 'f1': 0.6175902462778716, 'f1_weighted': 0.6176573784185434, 'precision': 0.6319970142821564, 'precision_weighted': 0.6320172248665896, 'recall': 0.6178904428904429, 'recall_weighted': 0.6180104031209362, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6137841352405722, 'f1': 0.6114232194958468, 'f1_weighted': 0.6115373683754285, 'precision': 0.6261043824913668, 'precision_weighted': 0.6261513374585087, 'recall': 0.6136197136197136, 'recall_weighted': 0.6137841352405722, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.590702210663199, 'f1': 0.588271407909425, 'f1_weighted': 0.5883518793255766, 'precision': 0.599093003454499

In [12]:
import torch
del embedding_model, trainer, train_loss
torch.cuda.empty_cache()

In [14]:
from datasets import Dataset

mapping = {2: 0, 1: 0, 0: 1}

train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

In [18]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

args = SentenceTransformerTrainingArguments(
    output_dir="cosine_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.230419
200,0.174233
300,0.169125
400,0.160030
500,0.152568
600,0.159014
700,0.150260
800,0.155552
900,0.150645
1000,0.147770


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.15769024773888762, metrics={'train_runtime': 2536.1069, 'train_samples_per_second': 19.715, 'train_steps_per_second': 0.616, 'total_flos': 0.0, 'train_loss': 0.15769024773888762, 'epoch': 1.0})

In [19]:
evaluator(embedding_model)

{'pearson_cosine': 0.721767058390524, 'spearman_cosine': 0.7248471336767005}

In [20]:
import torch
del embedding_model, trainer, train_loss
torch.cuda.empty_cache()

In [22]:
import random
from tqdm import tqdm

mnli = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(50_000))
mnli = mnli.remove_columns("idx")

mnli = mnli.filter(lambda x: True if x["label"]==0 else False)

train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = list(mnli["hypothesis"])
random.shuffle(soft_negatives)

for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row['premise'])
    train_dataset["positive"].append(row['hypothesis'])
    train_dataset["negative"].append(soft_negative)

train_dataset = Dataset.from_dict(train_dataset)

16875it [00:00, 22894.83it/s]


In [24]:
embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.333839
200,0.106618
300,0.079281
400,0.065811
500,0.066983


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=528, training_loss=0.12739246451493466, metrics={'train_runtime': 349.3606, 'train_samples_per_second': 48.303, 'train_steps_per_second': 1.511, 'total_flos': 0.0, 'train_loss': 0.12739246451493466, 'epoch': 1.0})

In [25]:
evaluator(embedding_model)

{'pearson_cosine': 0.8078265158888895, 'spearman_cosine': 0.8108787891334734}